# SDF Utilities

`isoext.sdf` is a small toolbox of signed distance functions for
testing and demos. Nothing in the library requires it: extraction
only sees the values you put on the grid, however you compute them.
{doc}`grids` shows examples with raw PyTorch.


In [1]:
import torch
import isoext
from isoext.sdf import *
from isoext import viewer

grid = isoext.UniformGrid([128, 128, 128])


## Meshes

`TriangleMeshSDF` turns a triangle mesh into a signed distance function, so a
mesh can be sampled into a grid, edited or combined like any other
field, and extracted again.

```python
TriangleMeshSDF(vertices, faces, signed=True)
```

`isoext.assets.load_mesh` downloads a few well-known test meshes on
first use and caches them:

- `"armadillo"` ({cite:t}`krishnamurthy1996`), `"bunny"`
  ({cite:t}`turk1994`) and `"dragon"` ({cite:t}`curless1996`), courtesy
  of the Stanford Computer Graphics Laboratory, for research use
- `"spot"` from {cite:t}`crane2012`, public domain

The armadillo, sampled on a 256 cell grid and extracted again:

In [2]:
vertices, faces = isoext.assets.load_mesh("armadillo")
armadillo = TriangleMeshSDF(vertices, faces)

fine = isoext.UniformGrid([256, 256, 256])
fine.set_values(armadillo(fine.get_points()))
v, f = isoext.dual_marching_cubes(fine)
print(f"{len(faces):,} mesh triangles in, {len(f):,} out")
viewer.embed(v, f, color="lightsteelblue")

345,944 mesh triangles in, 204,664 out


The distance is the distance to the closest triangle, found through a
bounding volume hierarchy on the GPU ({cite:t}`ericson2005`;
{cite:t}`jones2006` survey the approaches). The sign is the generalized
winding number ({cite:t}`jacobson2013`), evaluated with the tree-based
approximation of {cite:t}`barill2018`, so meshes with holes like the
bunny and the dragon still get a sensible sign. `sign="parity"` counts
ray crossings instead ({cite:t}`nooruddin2003`), which needs a closed
mesh.

The gradient is the direction away from the closest point, so
`get_sdf_normal` and `project_to_surface` work on a mesh SDF, and
`closest_points` returns the closest point on the mesh and its triangle
directly:

In [4]:
p = torch.tensor([[0.0, 0.0, 0.9], [0.5, 0.5, 0.5]], device="cuda")
q, face = armadillo.closest_points(p)
print("distances:", [round(d, 4) for d in armadillo(p).tolist()])
print("closest points:", [[round(x, 4) for x in row] for row in q.tolist()])
print("faces:", face.tolist())

distances: [0.1756, 0.1375]
closest points: [[-0.01, -0.0093, 0.725], [0.6087, 0.4526, 0.5696]]
faces: [281187, 12731]


## Primitives

### SphereSDF
```python
SphereSDF(radius: float)
```


In [5]:
grid = isoext.UniformGrid([128, 128, 128])

sphere = SphereSDF(radius=0.7)
grid.set_values(sphere(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f)

### TorusSDF
```python
TorusSDF(R: float, r: float)  # R=major radius, r=tube radius
```


In [6]:
torus = TorusSDF(R=0.6, r=0.2)
grid.set_values(torus(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="gold")

### CuboidSDF
```python
CuboidSDF(size: list[float])  # Full size in [x, y, z]
```


In [7]:
cube = CuboidSDF(size=[1.0, 1.0, 1.0])
grid.set_values(cube(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="salmon")

### MandelbulbSDF

A distance estimator for the Mandelbulb fractal of
{cite:t}`white2009`. The values approximate
the distance to the surface rather than being an exact SDF; fewer
iterations give a smoother shape.


In [8]:
bulb = MandelbulbSDF(iterations=6)

# The bulb needs slightly larger bounds than the shared grid above
bulb_grid = isoext.UniformGrid([128, 128, 128], aabb_min=[-1.2, -1.2, -1.2], aabb_max=[1.2, 1.2, 1.2])
bulb_grid.set_values(bulb(bulb_grid.get_points()))
v, f = isoext.marching_cubes(bulb_grid)
viewer.embed(v, f, color="coral")

## CSG Operations

Combine shapes using Constructive Solid Geometry:

| Operation | Description |
|-----------|-------------|
| `UnionOp([...])` | Combine shapes (min of SDFs) |
| `IntersectionOp([...])` | Keep overlap (max of SDFs) |
| `NegationOp(sdf)` | Invert inside/outside |
| `SmoothUnionOp([...], k)` | Smooth blend with radius k |


In [9]:
# Sphere with a hole drilled through it
sphere = SphereSDF(radius=0.7)
hole = CuboidSDF(size=[0.3, 0.3, 2.0])
drilled = IntersectionOp([sphere, NegationOp(hole)])

grid.set_values(drilled(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="orchid")

## Transformations

| Transform | Description |
|-----------|-------------|
| `TranslationOp(sdf, offset)` | Move by `[x, y, z]` |
| `RotationOp(sdf, axis, angle)` | Rotate around axis (degrees by default) |


In [10]:
# Two spheres with smooth blending
s1 = TranslationOp(SphereSDF(radius=0.4), offset=[-0.3, 0, 0])
s2 = TranslationOp(SphereSDF(radius=0.4), offset=[0.3, 0, 0])
blended = SmoothUnionOp([s1, s2], k=0.15)

grid.set_values(blended(grid.get_points()))
v, f = isoext.marching_cubes(grid)
viewer.embed(v, f, color="tomato")

## References

```{bibliography}
:filter: docname in docnames
```
